# Tugas MLOps P6 Praktikum MLOps

Nama: Anadia Carana

NIM: 123450019

Kelas: RB

## 1. Ekstraksi dan Penyiapan Direktori Kerja

In [1]:
from google.colab import files
import zipfile, os

up = files.upload()
name = next(iter(up))
zipfile.ZipFile(name).extractall('.')
os.chdir('mlops-praktikum')
print(os.getcwd())
print(sorted(os.listdir('.')))
assert 'src' in os.listdir('.'), "Folder src tidak ditemukan. Cek struktur zip."

Saving mlops-praktikum.zip to mlops-praktikum.zip
/content/mlops-praktikum
['.gitignore', 'README.md', 'configs', 'data', 'report', 'requirements.txt', 'src']


## 2. Instalasi Dependensi Perangkat Lunak

In [2]:
!pip install -q mlflow optuna dvc pyyaml scikit-learn
import torch, torchvision, mlflow, optuna
print('torch', torch.__version__, '| torchvision', torchvision.__version__)
print('mlflow', mlflow.__version__, '| optuna', optuna.__version__)
print('GPU tersedia:', torch.cuda.is_available())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.1/51.1 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 96.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 125.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 96.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.1/470.1 kB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.3/79.3 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 451.2/451.2 kB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 3. Inisialisasi Version Control (Git) dan Data Versioning (DVC)

In [3]:
!git init -q
!git config user.name "colab"
!git config user.email "colab@example.com"
!dvc init -q
!git add -A && git commit -qm "init"
!git log --oneline

070a5ba (HEAD -> master) init


## 4. Penyiapan Dataset dan Pelacakan (Tracking) DVC

In [5]:
!python src/prepare_data.py
!dvc add data/subset
!git add data/subset.dvc data/.gitignore && git commit -qm "track dataset"
!dvc status
!cat data/subset.dvc

100% 26.4M/26.4M [00:02<00:00, 10.2MB/s]
100% 29.5k/29.5k [00:00<00:00, 170kB/s]
100% 4.42M/4.42M [00:01<00:00, 3.22MB/s]
100% 5.15k/5.15k [00:00<00:00, 26.0MB/s]
Tersimpan di /content/mlops-praktikum/data/subset | train/val/test = 10000 2000 2000
⠋ Checking graph
Adding...:   0% 0/1 [00:00<?, ?file/s{'info': ''}]
!
          |0.00 [00:00,     ?file/s]
                                    
!
  0% |          |0/? [00:00<?,    ?files/s]
                                           
Adding data/subset to cache:   0% 0/6 [00:00<?, ?file/s]
Adding data/subset to cache:   0% 0/6 [00:00<?, ?file/s{'info': ''}]
                                                                    
!
          |0.00 [00:00,    ?files/s]
Adding...: 100% 1/1 [00:00<00:00, 13.79file/s{'info': ''}]

To track the changes with git, run:

	git add data/.gitignore data/subset.dvc

To enable auto staging, run:

	dvc config core.autostage true
Data and pipelines are up to date.
outs:
- md5: da2f2d32be4523ff3f9ad73d70f637dd.di

## 5. Konfigurasi dan Pelaksanaan MLflow Tracking Server

In [6]:
import subprocess, time, requests

server = subprocess.Popen(
    ["mlflow", "server",
     "--backend-store-uri", "sqlite:///mlflow.db",
     "--default-artifact-root", "./mlartifacts",
     "--host", "127.0.0.1", "--port", "5000"],
    stdout=open("mlflow_server.log", "w"), stderr=subprocess.STDOUT)

for i in range(40):
    try:
        if requests.get("http://127.0.0.1:5000/health", timeout=2).status_code == 200:
            print("Server MLflow siap.")
            break
    except Exception:
        time.sleep(2)
else:
    print(open("mlflow_server.log").read()[-3000:])
    raise RuntimeError("Server MLflow gagal menyala. Lihat log di atas.")

Server MLflow siap.


## 6. Pelatihan Model Baseline (Evaluasi Multi-Seed)

In [7]:
!python src/train_baseline.py

2026/10/10 12:59:58 INFO mlflow.tracking.fluent: Experiment with name 'fmnist-cnn' does not exist. Creating a new experiment.
2026/10/10 13:00:07 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu130) contains a local version label (+cu130). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/10/10 13:00:15 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmpjw3z69fy/model/data, flavor: pytorch). Fall back to return ['torch==2.11.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 
/usr/local/lib/python3.13/dist-packages/torch/export/pt2_archive/_package.py:790: UserWarning: The given buffer is not writable, and PyTorch does not support non-writable tensors. This means you can wri

## 7. Optimisasi Hyperparameter menggunakan Optuna (20 Trials)


In [8]:
!python src/tune_optuna.py --n-trials 20

[I 2026-10-10 13:01:46,428] A new study created in memory with name: no-name-87463b28-443c-4af4-aca6-544b504be2cf
🏃 View run trial-00 at: http://127.0.0.1:5000/#/experiments/1/runs/190812e11be4423b8395ab0fbd557d8f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
[I 2026-10-10 13:02:11,746] Trial 0 finished with value: 0.8745 and parameters: {'lr': 0.0005611516415334506, 'batch_size': 64, 'dropout': 0.17800932022121826, 'base_channels': 64, 'weight_decay': 6.358358856676247e-05}. Best is trial 0 with value: 0.8745.
🏃 View run trial-01 at: http://127.0.0.1:5000/#/experiments/1/runs/21343c2e74fc4a95b6a79117e5f274de
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
[I 2026-10-10 13:02:34,345] Trial 1 finished with value: 0.8835 and parameters: {'lr': 0.0026070247583707684, 'batch_size': 128, 'dropout': 0.20616955533913808, 'base_channels': 64, 'weight_decay': 3.752055855124284e-05}. Best is trial 1 with value: 0.8835.
🏃 View run trial-02 at: http://127.0.0.1:5000/#/exp

## 8. Evaluasi Konfigurasi Optimal (Multi-Seed)


In [9]:
!python src/final_seeds.py

2026/10/10 13:09:46 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu130) contains a local version label (+cu130). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/10/10 13:09:55 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmp5rlmpi_d/model/data, flavor: pytorch). Fall back to return ['torch==2.11.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 
/usr/local/lib/python3.13/dist-packages/torch/export/pt2_archive/_package.py:790: UserWarning: The given buffer is not writable, and PyTorch does not support non-writable tensors. This means you can write to the underlying (supposedly non-writable) buffer using the tensor. You may want to copy the buffer to protect its data or

## 9. Ablasi (4 varian x 3 seed)

In [10]:
!python src/ablation.py

🏃 View run ablation-full-seed11 at: http://127.0.0.1:5000/#/experiments/1/runs/d045a1e8f373408c86df54a4ebfed0e6
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run ablation-full-seed22 at: http://127.0.0.1:5000/#/experiments/1/runs/9b37ceb3ebf84a1fa701944779ba0b44
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run ablation-full-seed33 at: http://127.0.0.1:5000/#/experiments/1/runs/772b4702b5fe40f98755b2623f746137
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run ablation-full at: http://127.0.0.1:5000/#/experiments/1/runs/c504dc36c5654c9cad08fc0e5d42e4c4
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
full               val_acc = 0.8527 ± 0.0118
🏃 View run ablation-no_augmentation-seed11 at: http://127.0.0.1:5000/#/experiments/1/runs/a603abd89fcb447d8995f872219bc99a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run ablation-no_augmentation-seed22 at: http://127.0.0.1:5000/#/experiments/1/runs/e

## 10. Registrasi model dan alias `@champion`

In [11]:
!python src/register_champion.py

Successfully registered model 'fmnist-cnn-classifier'.
2026/10/10 13:15:57 WARNING mlflow.tracking._model_registry.fluent: Run with id 98e4b860e14e4b9e83feb6754663454c has no artifacts at artifact path 'model', registering model based on models:/m-cb80a50b24464658bc9c942b3a651009 instead
2026/10/10 13:15:57 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: fmnist-cnn-classifier, version 1
Created version '1' of model 'fmnist-cnn-classifier'.
Registered fmnist-cnn-classifier v1 dari run 98e4b860e14e4b9e83feb6754663454c -> alias @champion


## 11. Muat `models:/fmnist-cnn-classifier@champion` dan prediksi

In [12]:
!python src/predict_champion.py | tee report/predict_output.txt

/usr/local/lib/python3.13/dist-packages/torch/export/pt2_archive/_package.py:790: UserWarning: The given buffer is not writable, and PyTorch does not support non-writable tensors. This means you can write to the underlying (supposedly non-writable) buffer using the tensor. You may want to copy the buffer to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:1586.)
  tensor = torch.frombuffer(
Traceback (most recent call last):
  File "/content/mlops-praktikum/src/predict_champion.py", line 9, in <module>
    model = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@champion")
  File "/usr/local/lib/python3.13/dist-packages/mlflow/tracing/provider.py", line 1069, in wrapper
    return f(*args, **kwargs)
  File "/usr/local/lib/python3.13/dist-packages/mlflow/pyfunc/__init__.py", line 1171, in load_model
    model_impl = importlib.import

## 12. Ringkasan hasil

In [13]:
!python src/summarize.py
print("\n=== Hyperparameter terbaik ===")
!cat configs/best_config.json
print("\n=== Hash data (md5) ===")
!grep md5 data/subset.dvc

# Hasil (otomatis dari MLflow)


## Baseline (3 seed, validation)

| Run | val_acc | val_macro_f1 |
|---|---|---|
| baseline-cnn | 0.8830 ± 0.0026 | 0.8818 ± 0.0043 |

## Konfigurasi terbaik (3 seed)

| Run | val_acc | val_macro_f1 | test_acc | test_macro_f1 |
|---|---|---|---|---|
| final-best-config | 0.8542 ± 0.0035 | 0.8503 ± 0.0039 | 0.8600 ± 0.0048 | 0.8579 ± 0.0046 |

## Ablasi (3 seed, validation)

| Run | val_acc | val_macro_f1 |
|---|---|---|
| ablation-no_batchnorm | 0.8637 ± 0.0123 | 0.8610 ± 0.0147 |
| ablation-no_dropout | 0.8670 ± 0.0064 | 0.8648 ± 0.0088 |
| ablation-no_augmentation | 0.8810 ± 0.0036 | 0.8792 ± 0.0048 |
| ablation-full | 0.8527 ± 0.0118 | 0.8478 ± 0.0119 |

## Tuning

Jumlah trial tercatat: 20


=== Hyperparameter terbaik ===
{
  "lr": 0.0026070247583707684,
  "batch_size": 128,
  "epochs": 10,
  "dropout": 0.20616955533913808,
  "use_bn": true,
  "augment": true,
  "base_channels": 64,
  "weight_decay": 3.752055855124284e-05,
  "seed": 42
}
=== Hash da

## 13. Kemas hasil dan unduh

In [14]:
try:
    server.terminate(); server.wait(timeout=10)
except Exception as e:
    print("Server sudah berhenti:", e)

!git add -A && git commit -qm "results" ; true
!zip -qr hasil-mlops.zip mlflow.db mlartifacts configs/best_config.json report/results.md report/predict_output.txt data/subset.dvc data/.gitignore
!ls -lh hasil-mlops.zip
from google.colab import files
files.download('hasil-mlops.zip')

-rw-r--r-- 1 root root 15M Oct 10 13:17 hasil-mlops.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 14. Akses MLflow UI via Cloudflare Tunnel

In [15]:
import os, subprocess, time, re, requests
os.chdir('/content/mlops-praktikum')

# Jalankan hanya kalau runtime Colab sempat reset (file hasil hilang):
# !pip install -q mlflow
# from google.colab import files; up = files.upload()   # pilih hasil-mlops.zip
# import zipfile; zipfile.ZipFile(next(iter(up))).extractall('.')

server = subprocess.Popen(
    ["mlflow", "server", "--backend-store-uri", "sqlite:///mlflow.db",
     "--default-artifact-root", "./mlartifacts", "--host", "127.0.0.1", "--port", "5000"],
    stdout=open("mlflow_server.log", "w"), stderr=subprocess.STDOUT)
for _ in range(40):
    try:
        if requests.get("http://127.0.0.1:5000/health", timeout=2).status_code == 200:
            print("Server siap"); break
    except Exception:
        time.sleep(2)

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
tun = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:5000",
                        "--http-host-header", "127.0.0.1:5000"],
                       stdout=open("tunnel.log", "w"), stderr=subprocess.STDOUT)
for _ in range(30):
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("tunnel.log").read())
    if m:
        print("BUKA DI BROWSER:", m.group(0)); break
    time.sleep(2)

Server siap
BUKA DI BROWSER: https://workflow-fisheries-winston-knock.trycloudflare.com


## 15. Verifikasi Data Run MLflow secara Programatik

In [16]:
print(open('/content/mlops-praktikum/mlflow_server.log').read()[-3000:])

import mlflow
mlflow.set_tracking_uri("http://127.0.0.1:5000")
df = mlflow.search_runs(experiment_names=["fmnist-cnn"])
print("Jumlah run:", len(df))
print(df["tags.stage"].value_counts())

ET /static-files/static/css/5343.b4cc2aa3.chunk.css HTTP/1.1" 200 OK
2026/10/10 13:30:15 INFO:     103.126.172.129:0 - "GET /static-files/static/js/9867.fe03ab1c.chunk.js HTTP/1.1" 200 OK
2026/10/10 13:30:15 INFO:     103.126.172.129:0 - "GET /static-files/static/js/4751.94bc6db7.chunk.js HTTP/1.1" 200 OK
2026/10/10 13:30:15 INFO:     103.126.172.129:0 - "GET /static-files/static/js/3878.0467b2c4.chunk.js HTTP/1.1" 200 OK
2026/10/10 13:30:15 INFO:     103.126.172.129:0 - "GET /static-files/static/js/8503.9d34794c.chunk.js HTTP/1.1" 200 OK
2026/10/10 13:30:15 INFO:     103.126.172.129:0 - "GET /static-files/static/js/5343.bbb1f624.chunk.js HTTP/1.1" 200 OK
2026/10/10 13:30:15 INFO:     103.126.172.129:0 - "GET /static-files/static/js/2369.90891e4f.chunk.js HTTP/1.1" 200 OK
2026/10/10 13:30:15 INFO:     103.126.172.129:0 - "GET /static-files/static/js/855.aefc6d74.chunk.js HTTP/1.1" 200 OK
2026/10/10 13:30:15 INFO:     103.126.172.129:0 - "GET /static-files/static/js/3121.3be60de2.chunk.

In [17]:
!mlflow server --help | grep -i -E "cors|allowed-hosts|security"

      ...add --allowed-hosts host.co --cors-allowed-origins https://host.co
    Security  --allowed-hosts, --cors-allowed-origins, --x-frame-options, ...
                                  network. NOTE: This is NOT a security
                                  --allowed-hosts.
  --allowed-hosts TEXT            Comma-separated list of allowed Host headers
  --cors-allowed-origins TEXT     Comma-separated list of allowed CORS origins
                                  CORS attacks allow malicious websites to
  --disable-security-middleware   DANGEROUS: Disable all security middleware
                                  including CORS protection and host
                                  validation. This completely removes security
                                  is vulnerable to CORS attacks, DNS
                                  configuring specific security settings with
                                  --cors-allowed-origins and --allowed-hosts.


In [18]:
import subprocess, time, requests, os
os.chdir('/content/mlops-praktikum')

subprocess.run(["pkill", "-f", "mlflow server"])
time.sleep(3)

ORIGIN = "https://workflow-fisheries-winston-knock.trycloudflare.com"
server = subprocess.Popen(
    ["mlflow", "server", "--backend-store-uri", "sqlite:///mlflow.db",
     "--default-artifact-root", "./mlartifacts", "--host", "127.0.0.1", "--port", "5000",
     "--cors-allowed-origins", ORIGIN],
    stdout=open("mlflow_server.log", "w"), stderr=subprocess.STDOUT)

for _ in range(40):
    try:
        if requests.get("http://127.0.0.1:5000/health", timeout=2).status_code == 200:
            print("Server siap"); break
    except Exception:
        time.sleep(2)
else:
    print(open("mlflow_server.log").read()[-2000:])

Server siap


In [19]:
import mlflow
mlflow.set_tracking_uri("http://127.0.0.1:5000")
df = mlflow.search_runs(experiment_names=["fmnist-cnn"])
s = df[df["tags.stage"].str.endswith("-seed", na=False)]
cols = ["tags.mlflow.runName", "metrics.val_acc", "metrics.train_loss", "metrics.val_loss"]
print(s[cols].sort_values("tags.mlflow.runName").to_string(index=False))
t = df[df["tags.stage"] == "tuning"]
print(t[["metrics.best_val_acc"]])

            tags.mlflow.runName  metrics.val_acc  metrics.train_loss  metrics.val_loss
           ablation-full-seed11           0.8405            0.492873          0.410164
           ablation-full-seed22           0.8535            0.501121          0.396754
           ablation-full-seed33           0.8640            0.444545          0.392130
ablation-no_augmentation-seed11           0.8840            0.246840          0.323564
ablation-no_augmentation-seed22           0.8770            0.277688          0.363155
ablation-no_augmentation-seed33           0.8820            0.227242          0.341820
   ablation-no_batchnorm-seed11           0.8765            0.358719          0.325973
   ablation-no_batchnorm-seed22           0.8520            0.406444          0.390628
   ablation-no_batchnorm-seed33           0.8625            0.344049          0.374352
     ablation-no_dropout-seed11           0.8725            0.382545          0.339914
     ablation-no_dropout-seed22           0

## 16. Troubleshooting Inferensi Model Champion (Penanganan Ekspor GPU ke CPU)

In [21]:
import os; os.chdir('/content/mlops-praktikum')
!CUDA_VISIBLE_DEVICES="" python src/predict_champion.py 2>&1 | tee report/predict_output.txt

/usr/local/lib/python3.13/dist-packages/torch/export/pt2_archive/_package.py:790: UserWarning: The given buffer is not writable, and PyTorch does not support non-writable tensors. This means you can write to the underlying (supposedly non-writable) buffer using the tensor. You may want to copy the buffer to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:1586.)
  tensor = torch.frombuffer(
[0] prediksi=Coat         | label asli=Coat
[1] prediksi=Ankle boot   | label asli=Ankle boot
[2] prediksi=Coat         | label asli=Coat
[3] prediksi=Coat         | label asli=Coat
[4] prediksi=Ankle boot   | label asli=Ankle boot
[5] prediksi=T-shirt/top  | label asli=T-shirt/top
[6] prediksi=Trouser      | label asli=Trouser
[7] prediksi=Trouser      | label asli=Trouser
[8] prediksi=Sandal       | label asli=Sandal
[9] prediksi=Pullover    